# Phase 1: Train / Test / Validation Split

This is the first phase of building out the paper replication into a full experiment: splitting
the dataset into three separate, non-overlapping sets before any model development begins.

- **Training set (70%)** — used to fit models.
- **Test set (20%)** — used repeatedly during development to debug and compare models.
- **Validation set (10%)** — held out and **not touched** until the models built on
  Training/Test are bug-free and finalized. It is used exactly once, at the end, as an unbiased
  check of how the final model generalizes to unseen data.

Dataset: Kaggle ["Video Game Sales and Ratings"](https://www.kaggle.com/datasets/thedevastator/video-game-sales-and-rating),
same file and preprocessing used in the paper replication (`replicate_paper.ipynb`).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
CSV_PATH = "video_game_sales_and_ratings.csv"

TRAIN_FRAC = 0.70
TEST_FRAC = 0.20
VAL_FRAC = 0.10
assert abs(TRAIN_FRAC + TEST_FRAC + VAL_FRAC - 1.0) < 1e-9

## Load and clean the dataset

Same preprocessing as the paper replication: add the row `index` feature, restrict to the
paper's feature set + target, and drop rows with missing values.

In [2]:
df = pd.read_csv(CSV_PATH)
df = df.reset_index(drop=False)  # adds an "index" column: 0, 1, 2, ...

FEATURES = [
    "index",
    "Year_of_Release",
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales",
    "Critic_Score",
    "Critic_Count",
    "User_Count",
]
TARGET = "Global_Sales"

model_df = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)

print(f"Rows before cleaning: {len(df)}")
print(f"Rows after dropping missing values in features/target: {len(model_df)}")

Rows before cleaning: 16719
Rows after dropping missing values in features/target: 6894


## Split into Train / Test / Validation

Two-step split to hit exact 70/20/10 proportions of the full cleaned dataset:

1. Carve off the **Validation** set first (10%) — this set is set aside immediately and not
   examined further in this notebook.
2. Split the remaining 90% into **Training** (70% of the total) and **Test** (20% of the total).

In [3]:
# Step 1: set aside Validation (10%) — untouched from here on.
train_test_df, validation_df = train_test_split(
    model_df, test_size=VAL_FRAC, random_state=RANDOM_STATE
)

# Step 2: split the remaining 90% into Training (70%) and Test (20%) of the ORIGINAL total.
# 20% of the total is 20/90 of the remaining 90%; likewise 70% of the total is 70/90 of it.
relative_test_frac = TEST_FRAC / (TRAIN_FRAC + TEST_FRAC)
train_df, test_df = train_test_split(
    train_test_df, test_size=relative_test_frac, random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)

## Verify split sizes and proportions

In [4]:
total = len(model_df)
summary = pd.DataFrame({
    "Rows": [len(train_df), len(test_df), len(validation_df), total],
    "Proportion": [
        len(train_df) / total,
        len(test_df) / total,
        len(validation_df) / total,
        total / total,
    ],
}, index=["Training", "Test", "Validation", "Total"])
summary["Proportion"] = summary["Proportion"].map(lambda p: f"{p:.1%}")
summary

,Rows,Proportion
Training,4825,70.0%
Test,1379,20.0%
Validation,690,10.0%
Total,6894,100.0%


## Save the splits to disk

Each split is written to its own CSV under `splits/`, so later phases can load Training and Test
freely while the Validation file stays untouched until the model is finalized.

In [5]:
import os

os.makedirs("splits", exist_ok=True)
train_df.to_csv("splits/train.csv", index=False)
test_df.to_csv("splits/test.csv", index=False)
validation_df.to_csv("splits/validation.csv", index=False)

print("Saved splits/train.csv, splits/test.csv, splits/validation.csv")

Saved splits/train.csv, splits/test.csv, splits/validation.csv
